# LLM的内部机制

在了解了分词和词嵌入之后，我们已经准备好深入探究语言模型的工作原理了。在本章中，我们将探讨TransformerLLM的主要工作原理。我们将重点关注文本生成模型，以便更深入地理解生成式LLM。

我们将探讨相关概念并通过一些代码示例来演示这些概念。我们先加载一个语言模型，并定义流水线，以备生成文本。在第一次阅读时，你可以跳过代码部分，专注于理解概念。然后在第二次阅读时，通过代码来动手应用这些概念。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained("../models/Phi-3-mini-4k-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "../models/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
)
# 创建流水线
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## 3.1 Transformer模型概述

我们先了解Transformer的基础知识，再了解Transformer模型自2017年问世以来的改进。

### 3.1.1 已训练Transformer LLM的输入和输出

要理解Transformer LLM的行为，最常见的方式是将其视为一个接收文本输入并生成响应文本的软件系统。当一个足够大的文本输入-输出模型在足够大的高质量数据集上完成训练后，它就能够生成令人印象深刻且实用的输出。图3-1展示了一个用于撰写电子邮件的此类模型。

<div align="center">
 <img src="./picture/3-1.jpg">
 <p>图3-1：从整体来看，Transformer LLM接收提示词输入，并输出生成的文本</p>
</div>

模型并不是一次性生成所有文本，而是一次生成一个词元。图3-2展示了响应输入提示词时的四个词元生成步骤。每个词元生成步骤都是模型的一次前向传播（在机器学习中，前向传播指的是输入进入神经网络并流经计算图，最终在另一端产生输出所需的计算过程）​。

<div align="center">
  <img src="./picture/3-2.jpg">
  <p>图3-2：Transformer LLM一次生成一个词元，而不是一次生成全部文本</p>
</div>

在生成当前词元后，我们将输出词元追加到输入提示词的末尾，从而调整下一次生成的输入提示词。我们可以在图3-3中看到这一点。

<div align="center">
  <img src="./picture/3-3.jpg">
  <p>图3-3：输出词元被追加到提示词末尾后，得到的新文本会再次提供给模型进行另一次前向传播，以生成下一个词元</p>
</div>

这让我们对模型有了更准确的认识，它只是基于输入提示词预测下一个词元。神经网络外围的软件基本上就是在循环运行这个模型，按顺序扩展生成的文本，直到完成。

在机器学习中，有一个专门的词用来描述使用早期预测来进行后续预测的模型（例如，模型使用生成的第一个词元来生成第二个词元）​，这类模型被称为自回归模型(autoregressive model)。这就是为什么文本生成式LLM也被称为自回归模型。这一名称通常用于区分文本生成模型与像BERT这样的非自回归的文本表示模型。

如下所示，当我们使用LLM生成文本时，在底层发生的就是这种自回归的、逐词元生成的过程。

In [2]:
prompt = """Write an email apologizing to Sarah for the tragic gardening mishap.
Explain how it happened."""
output = generator(prompt)
print(output[0]['generated_text'])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.





## Response:Subject: Sincere Apologies for the Gardening Mishap

Dear Sarah,

I hope this email finds you well. I am writing to express my deepest apologies for the unfortun


我们可以看到模型从Subject（主题）一行开始写这封邮件。它突然停止，是因为达到了我们设置的max_new_tokens为50的限制。如果我们将这个值设置得更大，它会继续生成内容，直到完成这封邮件。

### 3.1.2 前向传播的组成

除了循环之外，前向传播还有两个关键的内部组件：分词器和语言建模头(language modeling head，LM head)。图3-4展示了这些组件在系统中的位置。在上一章中，我们看到了分词器如何将文本分解成词元ID序列，然后用作模型的输入。

分词器之后是神经网络，由一系列Transformer块堆叠而成，负责执行所有的处理工作。在这些堆叠的块之后是语言建模头，它将Transformer块的输出转换为预测下一个词元的概率分数。

<div align="center">
    <img src="./picture/3-4.jpg">
    <p>图3-4：Transformer LLM由分词器、堆叠的Transformer块和语言建模头组成</p>
</div>

回顾第2章的内容，分词器包含一个词元表，即分词器的词表。模型为词表中的每个词元都关联了一个向量表示（词元嵌入）​。图3-5展示了一个拥有50000个词元的词表及其对应的词元嵌入。

<div align="center">
    <img src="./picture/3-5.jpg">
    <p>图3-5：分词器拥有50000个词元的词表，模型为这些词元关联了词元嵌入</p>
</div>

计算流按照箭头方向从上到下进行。对于每个生成的词元，处理过程会按顺序依次经过堆叠成一列的所有Transformer块，然后到达语言建模头，最后输出下一个词元的概率分布，如图3-6所示。

<div align="center">
    <img src="./picture/3-6.jpg">
    <p>图3-6：在前向传播结束时，模型为词表中的每个词元预测一个概率分数</p>
</div>

语言建模头本身是一个简单的神经网络层。它可以连接到堆叠的Transformer块上的多种可能的“头”之一，用于构建不同类型的系统。其他类型的Transformer头包括序列分类头和词元分类头。

我们只需打印模型变量，就可以按顺序显示所有层。对于这个模型，我们得到：

In [3]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

观察这个结构，我们可以注意到以下几个重点。

· 这个结构展示了模型的各种嵌套层。模型的主要部分标记为model，随后是lm_head。

· 在Phi3Model内部，我们可以看到嵌入矩阵embed_tokens及其维度。它有32064个词元，每个词元的向量大小为3072。

· 暂时跳过dropout层，我们可以看到下一个主要组件是堆叠的Transformer解码器层。它包含32个Phi3DecoderLayer类型的块。

· 这些Transformer块中的每一个都包含一个注意力层和一个前馈神经网络（也称为MLP或多层感知器）​。我们将在本章后面详细介绍这些内容。

· 最后，我们看到lm_head接收一个大小为3072的向量，并输出一个大小等于模型所知词元数量的向量。该输出是每个词元的概率分数，帮助我们选择输出词元

### 3.1.3 从概率分布中选择单个词元（采样/解码）

在处理结束时，模型会为词表中的每个词元输出一个概率分数，正如我们在图3-6中看到的那样。从概率分布中选择单个词元的方法称为解码策略。图3-7展示了示例是如何选择词元Dear的。

<div align="center">
    <img src="./picture/3-7.jpg">
    <p>图3-7：经过模型前向传播后，基于上下文，模型可能输出的概率最高的几个词元。我们的解码策略通过基于概率的采样来决定输出哪个词元</p>
</div>

最简单的解码策略就是始终选择概率分数最高的词元。但在实践中，对于大多数使用场景来说，这种方法往往无法产生最佳输出。一个更好的方法是引入一些随机性，有时选择概率第二高或第三高的词元。用统计学家的话来说，这种思想就是根据概率分数对概率分布进行采样。

对于图3-7中的例子来说，如果Dear作为下一个词元的概率为40%，那么它被选中的概率就是40%（而不是像贪心搜索那样，直接选择这个得分最高的词元）​。这样，其他词元也有机会根据其分数被选中。

每次都选择概率分数最高的词元的策略被称为贪心解码。这就是在LLM中将温度(temperature)参数设为零时会发生的情况。我们将在第6章讨论温度的概念。

让我们仔细看看演示这个过程的代码。在这个代码块中，我们将输入词元传递给模型，然后传给lm_head：

In [4]:
prompt = "The capital of France is"
# 对输入提示词进行分词
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
# 将词元ID移动到GPU上
input_ids = input_ids.to("cuda")
# 获取模型在lm_head之前的输出
model_output = model.model(input_ids)
# 获取lm_head的输出
lm_head_output = model.lm_head(model_output[0])

现在，lm_head_output的形状是[1, 5, 32064]​。我们可以使用lm_head_output[0,-1]来访问最后生成的词元的概率分数，其中索引0用于批次维度，表示一批数据中的第一个，索引-1用于获取序列中的最后一个词元。现在我们得到了全部32064个词元的概率分数列表。接下来我们可以获取得分最高的词元ID，然后解码，以得到生成的输出词元的文本：

In [5]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

### 3.1.4 并行词元处理和上下文长度

Transformer最引人注目的特性之一是，它比之前的语言处理神经网络架构更适合并行计算。在文本生成中，观察每个词元是如何处理的，就能初步了解这一点。我们从上一章知道，分词器会将文本分解成词元。然后每个输入词元都会流经自己的计算路径（这种直觉有助于我们更轻松地理解）​。我们可以在图3-8中看到这些独立的计算流（也可以理解为处理路径）​。

<div align="center">
 <img src="./picture/3-8.jpg">
 <p>图3-8：每个词元都通过自己的计算流进行处理（之后我们会看到，它们在注意力步骤中会有一些交互）</p>
</div>